In [16]:
import os
import joblib
import librosa
import numpy as np

sample_rate = 22050

scaler = joblib.load(os.path.join("models","svm_scaler.gz"))
emotion_encoder = joblib.load(os.path.join("models","svm_emotion_encoder.pkl"))
intensity_encoder = joblib.load(os.path.join("models","svm_intensity_encoder.pkl"))
gender_encoder = joblib.load(os.path.join("models","svm_gender_encoder.pkl"))

multi_svm = joblib.load(os.path.join("models","svm_multi.pkl"))

In [17]:

def extract_features(file_path):
    nmfcc = 13
    y, _ = librosa.load(file_path, sr=sample_rate, mono=True)
    mfcc   = librosa.feature.mfcc(y=y, sr=sample_rate, n_mfcc=nmfcc)
    chroma = librosa.feature.chroma_stft(S=np.abs(librosa.stft(y)), sr=sample_rate)
    rms    = librosa.feature.rms(y=y)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sample_rate)
    zero_crossing = librosa.feature.zero_crossing_rate(y)
    return np.hstack([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        chroma.mean(axis=1), chroma.std(axis=1),
        rms.mean(), rms.std(),
        spectral_centroid.mean(), spectral_centroid.std(),
        zero_crossing.mean(), zero_crossing.std()
    ])

In [ ]:
import datetime
import re


def numeric_sort_key(filename):
    match = re.search(r"\d+", filename)
    if match:
        return (0, int(match.group()), filename.lower())
    return (1, float("inf"), filename.lower())


with open(f"test_logs/svm_{datetime.datetime.now()}_other.log", "w") as log:
    out = ""
    for (dirp, _, files) in os.walk("test_file"):
        files.sort(key=numeric_sort_key)
        for f in files:
            out += f"\nFile: {f}\n"
            feature = extract_features(os.path.join(dirp, f))
            feature_scaled = scaler.transform([feature])
            predictions = multi_svm.predict(feature_scaled)[0]

            predicted_emotion = emotion_encoder.inverse_transform([predictions[0]])[0]
            predicted_intensity = intensity_encoder.inverse_transform([predictions[1]])[0]
            predicted_gender = gender_encoder.inverse_transform([predictions[2]])[0]

            out += f"\nPredicted Emotion: {predicted_emotion}\n"
            out += f"Predicted Intensity: {predicted_intensity}\n"
            out += f"Predicted Gender: {predicted_gender}\n"

    log.write(out)